# Robustness, latency, and memory analysis

**Goal:** test whether retrieval augmentation is robust when the external memory is imperfect.

We compare conditions:

1. **fresh**: normal retrieval.
2. **missing_evidence**: gold supporting documents are excluded from retrieval.
3. **noisy_context**: random irrelevant chunks are inserted into context.
4. **conflicting_context**: retrieved context is corrupted by replacing the gold answer with a wrong answer when possible.

This gives a controlled approximation of outdated or unreliable external knowledge.

In [1]:
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/polinakorobeinikova/IU/NLP/case-study/case-study-rag-factual-consistency


In [3]:
import json
import random
from pathlib import Path
import re

import pandas as pd
from tqdm.auto import tqdm

from src.config import (
    PROCESSED_DIR, INDEX_DIR, PREDICTIONS_DIR, METRICS_DIR,
    GENERATOR_MODEL, EMBEDDING_MODEL, RERANKER_MODEL, RANDOM_SEED
)
from src.data_utils import load_gold_doc_ids
from src.generation import (
    load_generator, build_rag_prompt,
    generate_answer, answer_loss_and_perplexity, estimate_model_size_mb
)
from src.metrics import (
    evaluate_qa_predictions, recall_at_k, support_recall_at_k, numeric_summary
)
from src.retrieval import (
    DenseRetriever, BM25Retriever, HybridRerankRetriever,
    format_context, retrieved_doc_ids
)
from src.analysis_utils import Timer, file_size_mb, directory_size_mb, save_json

pd.set_option("display.max_colwidth", 160)
random.seed(RANDOM_SEED)

In [4]:
questions_df = pd.read_parquet(PROCESSED_DIR / "questions.parquet")
corpus_df = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")

EVAL_N = min(2000, len(questions_df))
TOP_K = 5

eval_df = questions_df.head(EVAL_N).copy()
print(eval_df.shape)

(2000, 7)


In [5]:
dense_retriever = DenseRetriever(
    index_path=INDEX_DIR / "dense_faiss.index",
    doc_ids_path=INDEX_DIR / "dense_doc_ids.json",
    corpus_df=corpus_df,
    model_name=EMBEDDING_MODEL,
)

bm25_retriever = BM25Retriever(
    bm25_path=INDEX_DIR / "bm25.pkl",
    corpus_df=corpus_df,
)

advanced_retriever = HybridRerankRetriever(
    dense=dense_retriever,
    bm25=bm25_retriever,
    reranker_name=RERANKER_MODEL,
    dense_k=50,
    bm25_k=50,
    rrf_k=60,
    rerank_k=20,
)

tokenizer, model, device = load_generator(GENERATOR_MODEL)
print("Device:", device)

Device: mps



## Context corruption helpers


In [6]:
def get_random_noise_chunks(corpus_df, n=2, exclude_doc_ids=None):
    exclude_doc_ids = exclude_doc_ids or set()
    pool = corpus_df[~corpus_df["doc_id"].isin(exclude_doc_ids)]
    sample = pool.sample(n=min(n, len(pool)), random_state=random.randint(0, 10_000))
    return sample.to_dict("records")


def corrupt_chunks_by_replacing_answer(chunks, gold_answer, wrong_answer):
    corrupted = []
    changed_any = False

    for ch in chunks:
        ch = dict(ch)
        text = str(ch.get("text", ""))
        if gold_answer and gold_answer.lower() in text.lower():
            # Case-insensitive replacement while keeping the implementation simple.
            pattern = re.compile(re.escape(gold_answer), flags=re.IGNORECASE)
            text2 = pattern.sub(str(wrong_answer), text)
            if text2 != text:
                changed_any = True
            ch["text"] = text2
        corrupted.append(ch)

    return corrupted, changed_any

In [7]:
def retrieve_for_condition(row, retriever, condition):
    gold_doc_ids = set(load_gold_doc_ids(row["gold_doc_ids"]))

    if condition == "missing_evidence":
        chunks = retriever.retrieve(row["question"], top_k=TOP_K, exclude_doc_ids=gold_doc_ids)
        return chunks, {"excluded_gold_docs": True, "context_changed": True}

    chunks = retriever.retrieve(row["question"], top_k=TOP_K)

    if condition == "noisy_context":
        ret_ids = set(retrieved_doc_ids(chunks))
        noise = get_random_noise_chunks(corpus_df, n=2, exclude_doc_ids=ret_ids)
        # Keep top-3 real chunks + 2 random noisy chunks, so context length remains comparable.
        chunks = chunks[:3] + noise
        random.shuffle(chunks)
        return chunks, {"added_noise_docs": len(noise), "context_changed": True}

    if condition == "conflicting_context":
        # Use the next answer as a controlled wrong answer.
        wrong_answer = questions_df.sample(1, random_state=random.randint(0, 10_000))["answer"].iloc[0]
        chunks, changed = corrupt_chunks_by_replacing_answer(chunks, row["answer"], wrong_answer)
        return chunks, {"wrong_answer": wrong_answer, "context_changed": changed}

    return chunks, {"context_changed": False}


## Run robustness experiment

To reduce runtime, this notebook evaluates vanilla and advanced RAG on a smaller subset.


In [8]:
systems = {
    "vanilla_rag_dense": dense_retriever,
    "advanced_rag_hybrid_rerank": advanced_retriever,
}

conditions = ["fresh", "missing_evidence", "noisy_context", "conflicting_context"]

rows = []

for system_name, retriever in systems.items():
    for condition in conditions:
        for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"{system_name} | {condition}"):
            gold_doc_ids = load_gold_doc_ids(row["gold_doc_ids"])

            with Timer() as rt:
                chunks, condition_meta = retrieve_for_condition(row, retriever, condition)
            retrieval_latency = rt.elapsed

            ret_doc_ids = retrieved_doc_ids(chunks)
            context = format_context(chunks)
            prompt = build_rag_prompt(row["question"], context)

            pred, gen_latency = generate_answer(
                tokenizer=tokenizer,
                model=model,
                prompt=prompt,
                max_new_tokens=48,
                temperature=0.0,
            )

            ppl_info = answer_loss_and_perplexity(
                tokenizer=tokenizer,
                model=model,
                prompt=prompt,
                answer=row["answer"],
            )

            rows.append({
                "system": system_name,
                "condition": condition,
                "sample_id": row["sample_id"],
                "question": row["question"],
                "answer": row["answer"],
                "prediction": pred,
                "answer_loss": ppl_info["answer_loss"],
                "answer_ppl": ppl_info["answer_ppl"],
                "answer_n_tokens": ppl_info["answer_n_tokens"],
                "retrieved_doc_ids": json.dumps(ret_doc_ids, ensure_ascii=False),
                "gold_doc_ids": json.dumps(gold_doc_ids, ensure_ascii=False),
                "retrieval_hit_at_5": recall_at_k(ret_doc_ids, gold_doc_ids, 5),
                "support_recall_at_5": support_recall_at_k(ret_doc_ids, gold_doc_ids, 5),
                "retrieval_latency_sec": retrieval_latency,
                "generation_latency_sec": gen_latency,
                "total_latency_sec": retrieval_latency + gen_latency,
                **condition_meta,
            })

robust_df = pd.DataFrame(rows)
robust_df.to_csv(PREDICTIONS_DIR / "robustness_predictions.csv", index=False)

display(robust_df.head())


vanilla_rag_dense | fresh:   0%|          | 0/2000 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


vanilla_rag_dense | missing_evidence:   0%|          | 0/2000 [00:00<?, ?it/s]

vanilla_rag_dense | noisy_context:   0%|          | 0/2000 [00:00<?, ?it/s]

vanilla_rag_dense | conflicting_context:   0%|          | 0/2000 [00:00<?, ?it/s]

advanced_rag_hybrid_rerank | fresh:   0%|          | 0/2000 [00:00<?, ?it/s]

advanced_rag_hybrid_rerank | missing_evidence:   0%|          | 0/2000 [00:00<?, ?it/s]

advanced_rag_hybrid_rerank | noisy_context:   0%|          | 0/2000 [00:00<?, ?it/s]

advanced_rag_hybrid_rerank | conflicting_context:   0%|          | 0/2000 [00:00<?, ?it/s]

,system,condition,sample_id,question,answer,prediction,answer_loss,answer_ppl,answer_n_tokens,retrieved_doc_ids,gold_doc_ids,retrieval_hit_at_5,support_recall_at_5,retrieval_latency_sec,generation_latency_sec,total_latency_sec,context_changed,excluded_gold_docs,added_noise_docs,wrong_answer
0,vanilla_rag_dense,fresh,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,No.,5.074569,159.903219,1,"[""5a8b6d885542997f31a41d25::9::Ed Wood (film)"", ""5a8b57f25542995d1e6f1371::0::Ed Wood (film)"", ""5a84d2ea5542992a431d1aa8::0::Ed Wood"", ""5a8b57f25542995d1e6f...","[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b57f25542995d1e6f1371::4::Ed Wood""]",1.0,0.5,0.213026,2.407952,2.620977,False,NaN,NaN,NaN
1,vanilla_rag_dense,fresh,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,"To determine the government position held by the woman who portrayed Corliss Archer in the film Kiss and Tell, let's analyze the information step-by-step:",8.324216,4122.503440,3,"[""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)"", ""5ae1b1445542997283cd223d::7::A Kiss for Corliss"", ""5a8c7595554299585d9e36b6::5::A Kiss for Corli...","[""5a8c7595554299585d9e36b6::1::Shirley Temple"", ""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)""]",1.0,0.5,0.036360,2.085241,2.121601,False,NaN,NaN,NaN
2,vanilla_rag_dense,fresh,5a85ea095542994775f606a8,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,The Chronicles of Tornor,5.811562,334.140507,3,"[""5abeebe25542993fe9a41d9c::6::First North Americans"", ""5a73024f5542991f9a20c5fc::5::Multiverse: Exploring Poul Anderson's Worlds"", ""5a85ea095542994775f606a...","[""5a85ea095542994775f606a8::2::The Hork-Bajir Chronicles"", ""5a85ea095542994775f606a8::8::Animorphs""]",0.0,0.0,0.035522,3.006377,3.041898,False,NaN,NaN,NaN
3,vanilla_rag_dense,fresh,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,No.,3.145353,23.227881,1,"[""5adbf0a255429947ff17385a::6::Esma Sultan Mansion"", ""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5a88a35c554299206df2b316::2::Abidin Mosque"", ""5adbf0a255...","[""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5adbf0a255429947ff17385a::6::Esma Sultan Mansion""]",1.0,1.0,0.051678,2.284481,2.336159,False,NaN,NaN,NaN
4,vanilla_rag_dense,fresh,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City",New York City,2.243537,9.426616,6,"[""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)"", ""5ade15f35542997545bbbe47::0::Manhattan Romance"", ""5ae0bae555429906c02dab08::9::Stone Stanley Entertai...","[""5a8e3ea95542995a26add48d::3::Adriana Trigiani"", ""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)""]",1.0,0.5,0.040702,2.058586,2.099288,False,NaN,NaN,NaN


In [9]:
summary_rows = []

for (system, condition), group in robust_df.groupby(["system", "condition"]):
    qa = evaluate_qa_predictions(group.to_dict("records"))

    metrics = {
        "system": system,
        "condition": condition,
        **qa,
    }

    metrics.update(numeric_summary(group["answer_loss"], "answer_loss"))
    metrics.update(numeric_summary(group["answer_ppl"], "answer_ppl"))
    metrics.update(numeric_summary(group["retrieval_hit_at_5"], "retrieval_hit_at_5"))
    metrics.update(numeric_summary(group["support_recall_at_5"], "support_recall_at_5"))
    metrics.update(numeric_summary(group["retrieval_latency_sec"], "retrieval_latency_sec"))
    metrics.update(numeric_summary(group["generation_latency_sec"], "generation_latency_sec"))
    metrics.update(numeric_summary(group["total_latency_sec"], "total_latency_sec"))

    summary_rows.append(metrics)

robust_summary = pd.DataFrame(summary_rows)
robust_summary.to_csv(METRICS_DIR / "robustness_summary.csv", index=False)
display(robust_summary)

,system,condition,exact_match,token_f1,contains_answer,n,answer_loss_mean,answer_loss_median,answer_loss_p90,answer_loss_p95,...,generation_latency_sec_mean,generation_latency_sec_median,generation_latency_sec_p90,generation_latency_sec_p95,generation_latency_sec_max,total_latency_sec_mean,total_latency_sec_median,total_latency_sec_p90,total_latency_sec_p95,total_latency_sec_max
0,advanced_rag_hybrid_rerank,conflicting_context,0.0645,0.132087,0.1375,2000,4.026764,3.511364,8.355150,9.954699,...,2.536814,2.422021,2.975802,3.339010,13.106098,3.009535,2.882536,3.505620,3.888642,13.369888
1,advanced_rag_hybrid_rerank,fresh,0.1360,0.235866,0.3145,2000,2.666157,1.678559,6.414990,8.368244,...,2.564546,2.412807,3.234580,3.637258,6.508129,3.308101,3.028148,4.398575,4.984727,8.116201
2,advanced_rag_hybrid_rerank,missing_evidence,0.0615,0.115298,0.1455,2000,4.267008,3.701300,8.530205,10.304334,...,3.011904,2.692188,4.070971,4.383382,10.169812,3.515234,3.212772,4.666336,5.034014,10.597976
3,advanced_rag_hybrid_rerank,noisy_context,0.1180,0.209757,0.2855,2000,2.856370,1.893985,6.667561,8.666060,...,2.588806,2.437743,3.259087,3.561442,13.485506,3.068125,2.919789,3.801550,4.119870,14.006956
4,vanilla_rag_dense,conflicting_context,0.0570,0.114028,0.1255,2000,4.288275,3.748987,8.438060,10.093726,...,2.763603,2.594815,3.314122,3.694449,17.883134,2.817813,2.649321,3.391314,3.759585,17.900630
5,vanilla_rag_dense,fresh,0.1060,0.184893,0.2550,2000,3.245940,2.517337,7.283772,9.059860,...,2.665137,2.532079,3.156729,3.458903,14.248625,2.702310,2.568891,3.207044,3.513118,14.361630
6,vanilla_rag_dense,missing_evidence,0.0500,0.098896,0.1225,2000,4.499603,3.939072,8.764830,10.535580,...,2.815518,2.624837,3.396834,3.911420,23.401413,2.860821,2.665022,3.451040,3.971516,23.455936
7,vanilla_rag_dense,noisy_context,0.0940,0.164563,0.2270,2000,3.527675,2.814115,7.592297,9.554199,...,3.187079,2.782755,3.952624,5.142018,31.239624,3.306823,2.914005,4.109223,5.418776,31.401608



## Memory trade-off

We estimate memory/storage overhead from model parameters and index files.


In [10]:
memory_summary = {
    "generator_model": GENERATOR_MODEL,
    "generator_model_size_mb_estimate": estimate_model_size_mb(model),
    "dense_faiss_index_mb": file_size_mb(INDEX_DIR / "dense_faiss.index"),
    "dense_embeddings_mb": file_size_mb(INDEX_DIR / "dense_embeddings.npy"),
    "bm25_index_mb": file_size_mb(INDEX_DIR / "bm25.pkl"),
    "all_index_files_mb": directory_size_mb(INDEX_DIR),
    "n_documents": len(corpus_df),
}

save_json(METRICS_DIR / "memory_summary.json", memory_summary)
memory_summary

{'generator_model': 'Qwen/Qwen2.5-0.5B-Instruct',
 'generator_model_size_mb_estimate': 1884.58544921875,
 'dense_faiss_index_mb': 72.91117572784424,
 'dense_embeddings_mb': 72.9112548828125,
 'bm25_index_mb': 56.24495029449463,
 'all_index_files_mb': 212.5265769958496,
 'n_documents': 49774}

### Robustness analysis

The robustness experiment evaluates how retrieval-augmented generation behaves when the external document index is imperfect. The fresh condition represents normal retrieval. The missing-evidence condition removes supporting documents from the retrieved context. The noisy-context condition adds irrelevant distractor documents. The conflicting-context condition injects misleading context with an incorrect answer.

The results show that both vanilla and advanced RAG depend strongly on evidence quality. Under fresh retrieval, advanced RAG achieves `EM = 0.1360` and `token F1 = 0.2359`. When supporting evidence is removed, token F1 drops to `0.1153` and mean answer loss increases from `2.666` to `4.267`. Conflicting context also causes a strong degradation, reducing token F1 to `0.1321` and increasing loss to `4.027`.

Noisy context is less harmful than missing or conflicting evidence. For advanced RAG, token F1 decreases from `0.2359` to `0.2098`, and answer loss increases from `2.666` to `2.856`. This suggests that the model can sometimes ignore irrelevant context, but it is much more vulnerable when correct evidence is absent or contradicted.

Advanced RAG remains better than vanilla RAG across all robustness conditions. However, the degradation under missing and conflicting evidence shows that retrieval augmentation is not automatically factual. Its benefits depend on index freshness, evidence coverage, and retrieval quality.